# Person A — Notebook 1: ACDC Circuit Discovery
**CS 590NN | Amogh | Week 2 (Mar 17-23)**

Runs ACDC circuit discovery on 50 CounterFact samples and saves results to disk.

**After this notebook finishes:**
- Download `week2_circuit_log.json` from the Files panel
- Upload it to Notebook 2 before running the ablation

> Runtime: L4 GPU (Colab Pro)
> Do NOT run Notebook 2 in this same session — ACDC fills GPU memory.

### 1.0 Install
Run once. Runtime restarts automatically — do not re-run after restart.

In [ ]:
import subprocess, sys, os

def pip(args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + args)

pip(["numpy==1.26.4"])
pip(["transformer-lens","transformers","datasets","accelerate","einops","jaxtyping"])

print("Done. Restarting runtime...")
os.kill(os.getpid(), 9)

### 1.1 Imports
Start here after restart.

In [ ]:
import torch, json, re, requests
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from transformer_lens import HookedTransformer, ActivationCache

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    free, total = torch.cuda.mem_get_info()
    print(f"Memory : {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

import numpy as np
assert tuple(int(x) for x in np.__version__.split(".")[:2]) < (2,0),     "NumPy >= 2.0 — re-run cell 1.0 and restart."
print(f"NumPy  : {np.__version__}")

torch.manual_seed(42)
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
print("Imports OK")

Device : cuda
GPU    : NVIDIA L4
Memory : 23.5 GB free / 23.7 GB total
NumPy  : 1.26.4
Imports OK


### 1.2 Load Model

In [ ]:
MODEL_NAME_PRIMARY  = "Qwen/Qwen3-0.6B"
MODEL_NAME_FALLBACK = "Qwen/Qwen2.5-0.5B"

def load_model(name):
    return HookedTransformer.from_pretrained(
        name,
        center_unembed=True,
        center_writing_weights=False,
        fold_ln=True,
        refactor_factored_attn_matrices=False,
        device=DEVICE,
    )

try:
    model = load_model(MODEL_NAME_PRIMARY)
    MODEL_NAME = MODEL_NAME_PRIMARY
except Exception as e:
    print(f"Primary failed ({e}). Using fallback...")
    model = load_model(MODEL_NAME_FALLBACK)
    MODEL_NAME = MODEL_NAME_FALLBACK

model.eval()
model.tokenizer.pad_token = model.tokenizer.eos_token
free, total = torch.cuda.mem_get_info()
print(f"Loaded : {MODEL_NAME}")
print(f"Config : {model.cfg.n_layers} layers | {model.cfg.n_heads} heads | d_model={model.cfg.d_model}")
print(f"Memory : {free/1e9:.1f} GB free after load")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Loaded pretrained model Qwen/Qwen3-0.6B into HookedTransformer
Loaded : Qwen/Qwen3-0.6B
Config : 28 layers | 16 heads | d_model=1024
Memory : 20.3 GB free after load


### 1.3 Native ACDC Implementation

In [ ]:
class NativeACDC:
    """
    ACDC-style threshold-based circuit discovery via activation patching.
    Conmy et al., NeurIPS 2023. Implemented natively via TransformerLens.
    Activation caches are deleted after each sample to keep GPU memory flat.
    """
    def __init__(self, model, threshold=0.1):
        self.model     = model
        self.threshold = threshold

    def _logit_diff(self, logits, target_id, baseline_id):
        return (logits[0, -1, target_id] - logits[0, -1, baseline_id]).item()

    def run(self, prompt, target_true, target_new):
        true_id = self.model.to_tokens(target_true, prepend_bos=False)[0, 0].item()
        new_id  = self.model.to_tokens(target_new,  prepend_bos=False)[0, 0].item()
        tokens  = self.model.to_tokens(prompt)

        corrupt = tokens.clone()
        if tokens.shape[1] > 2:
            corrupt[0, 1:-1] = torch.randint(1000, self.model.cfg.d_vocab-1, (tokens.shape[1]-2,))

        with torch.no_grad():
            clean_logits,   clean_cache   = self.model.run_with_cache(tokens)
            corrupt_logits, corrupt_cache = self.model.run_with_cache(corrupt)

        clean_score   = self._logit_diff(clean_logits,   true_id, new_id)
        corrupt_score = self._logit_diff(corrupt_logits, true_id, new_id)
        effect_range  = abs(clean_score - corrupt_score) + 1e-8

        causal_scores = {}

        for layer in range(self.model.cfg.n_layers):
            # Attention heads
            hn = f"blocks.{layer}.attn.hook_result"
            if hn in clean_cache:
                for head in range(self.model.cfg.n_heads):
                    def make_attn(h=head, n=hn):
                        ca = corrupt_cache[n][:, :, h:h+1, :].clone()
                        def fn(v, hook): v[:, :, h:h+1, :] = ca; return v
                        return fn
                    with torch.no_grad():
                        pl = self.model.run_with_hooks(tokens, fwd_hooks=[(hn, make_attn())])
                    causal_scores[f"attn_{layer}_{head}"] = abs(self._logit_diff(pl, true_id, new_id) - clean_score) / effect_range

            # MLP layers
            hn = f"blocks.{layer}.hook_mlp_out"
            if hn in clean_cache:
                def make_mlp(n=hn):
                    ca = corrupt_cache[n].clone()
                    def fn(v, hook): return ca
                    return fn
                with torch.no_grad():
                    pl = self.model.run_with_hooks(tokens, fwd_hooks=[(hn, make_mlp())])
                causal_scores[f"mlp_{layer}"] = abs(self._logit_diff(pl, true_id, new_id) - clean_score) / effect_range

        # Free caches immediately
        del clean_cache, corrupt_cache, clean_logits, corrupt_logits
        torch.cuda.empty_cache()

        attn_heads, mlp_layers = [], []
        for node, score in causal_scores.items():
            if score >= self.threshold:
                parts = node.split("_")
                if parts[0] == "attn": attn_heads.append((int(parts[1]), int(parts[2])))
                else:                  mlp_layers.append(int(parts[1]))

        return {
            "attn_heads":    sorted(set(attn_heads)),
            "mlp_layers":    sorted(set(mlp_layers)),
            "causal_scores": causal_scores,
            "clean_score":   clean_score,
            "corrupt_score": corrupt_score,
        }

print("NativeACDC defined.")

NativeACDC defined.


### 1.4 Load CounterFact

In [ ]:
@dataclass
class EditSample:
    prompt:          str
    target_new:      str
    target_true:     str
    related_prompts: List[str] = field(default_factory=list)

print("Downloading CounterFact from ROME project source...")
raw_data = requests.get("https://rome.baulab.info/data/dsets/counterfact.json", timeout=60).json()
print(f"Downloaded {len(raw_data)} records")

N_SAMPLES = 50

def parse_counterfact(raw):
    neighborhood = raw.get("neighborhood_prompts", [])
    related = [p for p in neighborhood[:3] if isinstance(p, str)]
    return EditSample(
        prompt=raw["requested_rewrite"]["prompt"].format(raw["requested_rewrite"]["subject"]),
        target_new=" "  + raw["requested_rewrite"]["target_new"]["str"],
        target_true=" " + raw["requested_rewrite"]["target_true"]["str"],
        related_prompts=related,
    )

cf_samples = [parse_counterfact(raw_data[i]) for i in range(N_SAMPLES)]
print(f"Loaded {N_SAMPLES} samples")
print(f"Example: '{cf_samples[0].prompt}' -> true='{cf_samples[0].target_true}'")

Downloaded 21919 records
Loaded 50 samples
Example: 'The mother tongue of Danielle Darrieux is' -> true=' French'


### 1.5 Run ACDC on 50 Samples

In [ ]:
acdc        = NativeACDC(model, threshold=0.1)
circuit_log = []

print(f"Running ACDC on {N_SAMPLES} samples...")
print("Caches freed after each sample.\n")

for i, s in enumerate(cf_samples):
    result = acdc.run(s.prompt, s.target_true, s.target_new)
    circuit_log.append({
        "idx":          i,
        "prompt":       s.prompt,
        "attn_heads":   result["attn_heads"],
        "mlp_layers":   result["mlp_layers"],
        "clean_score":  round(result["clean_score"],    4),
        "corrupt_score":round(result["corrupt_score"],  4),
        "n_attn":       len(result["attn_heads"]),
        "n_mlp":        len(result["mlp_layers"]),
    })
    if i % 10 == 0:
        free = torch.cuda.mem_get_info()[0] / 1e9
        print(f"  [{i+1:>2}/{N_SAMPLES}]  attn={result['attn_heads'][:2]}  mlp={result['mlp_layers'][:3]}  gpu_free={free:.1f}GB")

with open(RESULTS_DIR / "week2_circuit_log.json", "w") as f:
    json.dump(circuit_log, f, indent=2)

from collections import Counter
all_attn = [str(h) for e in circuit_log for h in e["attn_heads"]]
all_mlp  = [str(l) for e in circuit_log for l in e["mlp_layers"]]
print(f"\nTop attention heads : {Counter(all_attn).most_common(5)}")
print(f"Top MLP layers      : {Counter(all_mlp).most_common(5)}")
print(f"Saved -> {RESULTS_DIR}/week2_circuit_log.json")

Running ACDC on 50 samples...
Caches freed after each sample.

  [ 1/50]  attn=[]  mlp=[0, 1, 2]  gpu_free=20.2GB
  [11/50]  attn=[]  mlp=[1, 2, 3]  gpu_free=20.2GB
  [21/50]  attn=[]  mlp=[0, 1, 2]  gpu_free=20.2GB
  [31/50]  attn=[]  mlp=[0, 1, 2]  gpu_free=20.2GB
  [41/50]  attn=[]  mlp=[0, 1, 2]  gpu_free=20.2GB

Top attention heads : []
Top MLP layers      : [('0', 48), ('4', 48), ('1', 44), ('3', 44), ('7', 42)]
Saved -> results/week2_circuit_log.json


### 1.6 Save and Download Results

In [ ]:
import shutil
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Summary
summary = {
    "author":    "Amogh",
    "notebook":  "Notebook 1 — ACDC Circuit Discovery",
    "model":     MODEL_NAME,
    "timestamp": timestamp,
    "n_samples": N_SAMPLES,
    "threshold": 0.1,
    "circuit_summary": {
        "avg_attn_per_sample": round(sum(e["n_attn"] for e in circuit_log) / N_SAMPLES, 2),
        "avg_mlp_per_sample":  round(sum(e["n_mlp"]  for e in circuit_log) / N_SAMPLES, 2),
        "top_attn_heads": Counter(all_attn).most_common(5),
        "top_mlp_layers": Counter(all_mlp).most_common(5),
    }
}

with open(RESULTS_DIR / f"summary_nb1_{timestamp}.json", "w") as f:
    json.dump(summary, f, indent=2)

zip_path = f"/content/PersonA_Notebook1_results_{timestamp}"
shutil.make_archive(zip_path, "zip", RESULTS_DIR)

print("=" * 55)
print(f"  NOTEBOOK 1 RESULTS — Amogh")
print(f"  {timestamp}")
print("=" * 55)
print(f"  Avg attn heads/sample : {summary['circuit_summary']['avg_attn_per_sample']}")
print(f"  Avg MLP layers/sample : {summary['circuit_summary']['avg_mlp_per_sample']}")
print(f"  Top heads             : {summary['circuit_summary']['top_attn_heads']}")
print()
print("  Files saved:")
for f in sorted(RESULTS_DIR.glob("*.json")):
    print(f"    {f.name:<40}  {f.stat().st_size//1024:>4} KB")
print(f"\n  Download: {zip_path}.zip")
print("=" * 55)

from google.colab import files
files.download(f"{zip_path}.zip")
print("\nDONE. Upload week2_circuit_log.json to Notebook 2.")

  NOTEBOOK 1 RESULTS — Amogh
  20260310_171841
  Avg attn heads/sample : 0.0
  Avg MLP layers/sample : 20.44
  Top heads             : []

  Files saved:
    summary_nb1_20260310_171841.json             0 KB
    week2_circuit_log.json                      20 KB

  Download: /content/PersonA_Notebook1_results_20260310_171841.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


DONE. Upload week2_circuit_log.json to Notebook 2.
